# Fix_W — corrected

Same method as `Fix_W.ipynb`, same output. Four things were wrong there:

| | `Fix_W.ipynb` | here |
|---|---|---|
| $H$ over land | `e3t_full.sum("z")` = 0, so $W_\eta = 0/0$ = NaN | masked out |
| below the seabed | `z` is NaN there, so `W_merc.fillna(0)` was undone by the subtraction | masked, then filled once at the end |
| output file | a 3-D `z` (256 MB) and 2-D `H` written as coordinates, and a coordinate named `z` sitting next to `depthw` | dropped |
| $\partial\eta/\partial t$ | `vovecrtz[depthw=0]` | switchable — see below |

## About $\partial\eta/\partial t$

`vovecrtz[depthw=0]` is **not** $\partial\eta/\partial t$. It is NEMO's full surface kinematic
boundary condition,

$$w\big|_{z=0} = \frac{\partial\eta}{\partial t} + \frac{P + R - E}{\rho_0},$$

and in the Amazon the runoff term dominates: it correlates only ~0.35 with the tendency of
`sossheig` and is about twice as large.

But that does not make it the wrong thing to subtract. The two choices do different jobs:

- `DETA_SOURCE = "vovecrtz"` — subtracts the whole surface velocity, so $W_{fixed}(0)=0$
  **exactly**. This is the rigid lid the derivation below promises, and it is what the original
  notebook did. What it removes is the free surface *and* the freshwater flux.
- `DETA_SOURCE = "ssh"` — subtracts only $\partial\eta/\partial t$, computed from `sossheig`.
  Physically the narrower statement, but then $W_{fixed}(0)$ is the freshwater flux, not zero,
  so the boundary condition in the derivation no longer holds.

If you want $W_{fixed}(0)=0$, use `"vovecrtz"` and describe it as the surface kinematic BC
rather than as $\partial\eta/\partial t$.

In [ ]:
year, month = 1993, 1

#   "vovecrtz" : subtract the full surface velocity  ->  W_fixed(0) = 0 exactly (rigid lid)
#   "ssh"      : subtract only deta/dt from sossheig ->  W_fixed(0) = freshwater flux
DETA_SOURCE = "vovecrtz"

In [ ]:
from pathlib import Path
import xarray as xr
import numpy as np
import os, subprocess, sys

In [ ]:
mesh_path = "Zgr_cmesh2.nc"
W_path    = f"W_{year}-{month:02d}.nc"
SSH_path  = f"SSH_{year}-{month:02d}c.nc"
U_path    = f"U_{year}-{month:02d}.nc"       # for the x/y coordinates only

outpath = "./"
W_out   = f"W_{year}-{month:02d}fc2.nc"      # rename to ...fc.nc to replace the old output

In [ ]:
## mesh load
ds_mesh = xr.open_dataset(mesh_path, chunks={})
ds_mesh = ds_mesh.assign_coords(x=np.arange(ds_mesh.sizes["x"]))
ds_mesh = ds_mesh.assign_coords(y=np.arange(ds_mesh.sizes["y"]))
ds_mesh = ds_mesh.assign_coords(z=np.arange(ds_mesh.sizes["z"]))
ds_mesh = ds_mesh.squeeze()

In [ ]:
#Construct W depth for each water filled cell
e3t_full = xr.where(
    (ds_mesh.z + 1) <= (ds_mesh.mbathy - 1),  # wet cells above bottom cells
    ds_mesh.e3t_0,  # fill with basin wide e3t for level,
    ds_mesh.e3t_ps,  # add partial cell height otherwise
).where((ds_mesh.z + 1) <= ds_mesh.mbathy)  # remove all non-wet cells below

In [ ]:
#W depths (top of cell) as vertical sum of the e3t (including the partially filled last cell
#above the bottom). Then rename the dimension z coming from the mesh file to depthw which
#we'll need to align with the W file later.

depthw_ps = e3t_full.cumsum("z").where((ds_mesh.z + 1) <= ds_mesh.mbathy)
depthw_ps = depthw_ps.shift(z=1).fillna(0.0)
depthw_ps = depthw_ps.where(ds_mesh.z <= ds_mesh.mbathy)
depthw_ps = depthw_ps.rename({"z": "depthw"}).drop_vars("depthw")

In [ ]:
## Calc height of water colum (relto $\eta=0$)
## FIX: sum of an all-NaN column is 0, which is a land point. Mask it so H is never 0.
H_bottom = e3t_full.sum("z")
H_bottom = H_bottom.where(H_bottom > 0)

In [ ]:
## load W file
ds_W = xr.open_dataset(W_path, chunks={"time_counter": 1})
ds_W = ds_W.assign_coords(x=np.arange(ds_W.sizes["x"]))
ds_W = ds_W.assign_coords(y=np.arange(ds_W.sizes["y"]))
ds_W = ds_W.assign_coords(z=-depthw_ps, H=H_bottom)

In [ ]:
## Load SSH (we need $\eta$)
## FIX: the time derivative needs the whole time axis in one chunk.
ds_SSH = xr.open_dataset(SSH_path, engine="netcdf4", chunks={"time_counter": -1})
ds_SSH = ds_SSH.assign_coords(x=np.arange(ds_SSH.sizes["x"]))
ds_SSH = ds_SSH.assign_coords(y=np.arange(ds_SSH.sizes["y"]))

## Correcting W for fixed sea surface:

$z$ is positive upward, $H$ is positive, and $\eta$ is positive upward

$$W_{merc}(z) = W_\eta(z) + W_{fixed}(z)$$

Normally,
$$W_\eta(z) = \frac{z+H}{\eta+H} \frac{\partial \eta}{\partial t}$$
but we register it to the surface and set.
$$W_\eta(z) \equiv \frac{z+H}{H} \frac{\partial \eta}{\partial t}$$
(Note the change in the denominator.)

Then
$$W_{fixed}(z) = W_{merc}(z) - W_\eta(z)$$
and
$$W_{fixed}(0) = W_{fixed}(-H) = 0$$

In [ ]:
eta = ds_SSH.sossheig

In [ ]:
w_surf  = ds_W.vovecrtz.isel(depthw=0, drop=True)          # deta/dt + (P+R-E)/rho0
eta_dot = eta.differentiate("time_counter", datetime_unit="s")   # deta/dt alone

deta_dt = w_surf if DETA_SOURCE == "vovecrtz" else eta_dot

In [ ]:
## how different are the two candidates?
_a = w_surf.values.ravel()
_b = eta_dot.values.ravel()
_g = np.isfinite(_a) & np.isfinite(_b)
print(f"rms vovecrtz[0]  (surface kinematic BC)  {np.sqrt((_a[_g]**2).mean()):.2e} m/s")
print(f"rms deta/dt      (from sossheig)         {np.sqrt((_b[_g]**2).mean()):.2e} m/s")
print(f"correlation between them                 {np.corrcoef(_a[_g], _b[_g])[0,1]:.3f}")
print(f"\nusing DETA_SOURCE = {DETA_SOURCE!r}")

In [ ]:
W_merc = ds_W.vovecrtz.rename("W_merc")

In [ ]:
## FIX: only defined where z and H are (i.e. in the water column); no 0/0 over land.
wet = np.isfinite(ds_W.z) & np.isfinite(ds_W.H)

W_eta = (ds_W.z + ds_W.H) / ds_W.H * deta_dt
W_eta = W_eta.rename("W_eta")
W_eta = W_eta.assign_coords(z=ds_W.z).where(wet)

In [ ]:
## FIX: mask first, fill once. Previously fillna(0) ran before a subtraction that reintroduced
## NaN everywhere z was undefined.
W_fixed = (W_merc.fillna(0.0) - W_eta.fillna(0.0)).where(wet, 0.0)
W_fixed = W_fixed.rename("W_fixed")

In [ ]:
## check the two boundary conditions the derivation promises
_t = min(1, W_fixed.sizes["time_counter"] - 1)
_s = W_fixed.isel(time_counter=_t, depthw=0).values
_mb = ds_mesh.mbathy.values.astype(int)
_b = W_fixed.isel(time_counter=_t).values[_mb.clip(0, 49),
                                          *np.indices(_mb.shape)]
print(f"max |W_fixed(z=0)|   {np.nanmax(np.abs(_s[_mb > 0])):.2e} m/s"
      "   <- zero only for DETA_SOURCE='vovecrtz'")
print(f"max |W_fixed(z=-H)|  {np.nanmax(np.abs(_b[_mb > 0])):.2e} m/s")

In [ ]:
## old coords
x = xr.open_dataset(U_path, chunks={}).x
y = xr.open_dataset(U_path, chunks={}).y

In [ ]:
#re-define the x and y to match with the U and V
W_fixed = W_fixed.assign_coords(x=x)
W_fixed = W_fixed.assign_coords(y=y)
W_fixed.name = "vovecrtz"
## FIX: drop the 3-D z and 2-D H coordinates so they are not written to the file
W_fixed = W_fixed.drop_vars([v for v in ("z", "H") if v in W_fixed.coords])
W_fixed = W_fixed.to_dataset()
W_fixed

In [ ]:
W_fixed = W_fixed.astype("float32")
W_fixed.to_netcdf(outpath + W_out,
                  encoding={"vovecrtz": dict(dtype="float32", zlib=False)})
print("written", outpath + W_out)

---

With `DETA_SOURCE = "vovecrtz"` both boundary conditions hold exactly: $W_{fixed}(0)=0$ at
every ocean point and $W_{fixed}(-H)=0$. With `"ssh"` only the bottom one does.

Worth knowing before you use this: the correction is small either way. The surface velocity
has an rms of ~2e-7 m/s, so $W_\eta$ moves a particle a median of a few centimetres over 30
days. It removes the free-surface signal from W, which is what it is for — it will not
visibly change trajectories.